# SafeNest 板端运行
将整个 board 文件夹上传到 PYNQ，打开此文件。先完成接线；本入口默认不连接 MQTT。避免同时运行其他占用 PL 的 Notebook。

In [ ]:
from pathlib import Path
from realtime_monitor import RealtimeMonitor
ROOT = Path.cwd()
assert (ROOT / 'safenest_audio.bit').is_file(), '请从 board 目录打开此 Notebook'
monitor = RealtimeMonitor(directory=ROOT, input_source='microphone',
    capture_seconds=3.0, save_raw=False, decision_threshold=0.25, log_batch_size=1)
monitor.hardware.fir.selftest(ROOT)
monitor.hw_alarm.selftest()
print('初始化与自检完成')

## 三轮真实录音测试
运行后真实录音并允许蜂鸣器报警；末轮等待2.2秒以保留完整鸣叫。

In [ ]:
import time
records = []
monitor.hw_alarm.arm(muted=False)
try:
    for i in range(3):
        print('开始第 %d 轮录音' % (i + 1), flush=True)
        row = monitor._process(monitor._capture(), pipelined=False)
        monitor._flush()
        records.append(row)
    time.sleep(2.2)
finally:
    monitor.hw_alarm.disarm()
    monitor._flush()
for r in records:
    print(r['segment_id'], r['class_name'], 'capture=', r['capture_seconds'],
          'post=', r['processing_seconds'], 'cnn=', r['model_seconds'],
          'score=', r['rolling_score'], 'alarm=', r['alarm_requested'])

日志生成在本目录 runtime/ 下。先复制需要保留的日志；原程序只保留当天结果。停止后不要继续复用发生过 DMA 错误的硬件对象。MQTT 参阅 report/mqtt_setup.md。